In [1]:
import os
import tensorflow as tf
import numpy as np

In [8]:
#############################################################################

# DATASET_PATH = r"D:\ANN and DL\aud\data"
DATASET_PATH = r"D:\ANN and DL\audio"

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Number of class folders:", len(os.listdir(DATASET_PATH)))


Dataset exists: True
Number of class folders: 1


In [9]:
#############################################################################
# LOAD AUDIO DATASET
train_set, test_set = tf.keras.utils.audio_dataset_from_directory(
    directory=DATASET_PATH,
    batch_size=16,
    validation_split=0.2,
    output_sequence_length=16000,
    seed=0,
    subset="both"
)

Found 3000 files belonging to 1 classes.
Using 2400 files for training.
Using 600 files for validation.


In [10]:
#############################################################################
# LABELS
label_names = np.array(train_set.class_names)
print("Total classes:", len(label_names))

Total classes: 1


In [11]:
#############################################################################
# REMOVE CHANNEL DIMENSION
def squeeze(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    return audio, label

train_set = train_set.map(squeeze, num_parallel_calls=tf.data.AUTOTUNE)
test_set  = test_set.map(squeeze, num_parallel_calls=tf.data.AUTOTUNE)

In [12]:
#############################################################################
# MEL-SPECTROGRAM USING PURE TENSORFLOW
def mel_spectrogram(audio, label):

    stft = tf.signal.stft(
        audio,
        frame_length=256,
        frame_step=128
    )
    spectrogram = tf.abs(stft)

    mel_filterbank = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=40,
        num_spectrogram_bins=spectrogram.shape[-1],
        sample_rate=16000,
        lower_edge_hertz=80.0,
        upper_edge_hertz=7600.0
    )

    mel_spec = tf.tensordot(spectrogram, mel_filterbank, 1)
    mel_spec = tf.math.log(mel_spec + 1e-6)

    return mel_spec, label

train_set = train_set.map(mel_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)
test_set  = test_set.map(mel_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)

In [13]:
#############################################################################
# PERFORMANCE
train_set = train_set.prefetch(tf.data.AUTOTUNE)
test_set  = test_set.prefetch(tf.data.AUTOTUNE)

#############################################################################
# FINAL SANITY CHECK
for x, y in train_set.take(1):
    print("Input shape:", x.shape)
    print("Labels shape:", y.shape)

Input shape: (16, 124, 40)
Labels shape: (16,)
